# Stage 1b — Galahad taxonomy v4 (Plaid PFCv1-aligned) with explicit flow

Upgrades labels v3 to a vendor taxonomy aligned with Plaid Personal Finance
Categories v1 (16 primary categories plus a detailed second level). Plaid PFC
is not an ISO standard, and PFCv2 is the current taxonomy for new Plaid
integrations as of December 2025. This executed notebook preserves the v1
checkpoint lineage rather than relabelling it after training. Key changes:

- **Inflow/outflow is first-class**: every row gets `flow` (inflow/outflow from
  the debit/credit flag) and direction-aware primaries — `TRANSFER_IN` vs
  `TRANSFER_OUT`, deposits vs withdrawals, loan repayments vs disbursements.
- `transaction_status` (reversal/unpaid/insufficient_funds) stays orthogonal.
- `possible_refund` flags spend-category rows arriving as credits.
- Merchants carry over from v3 canonicalisation unchanged.

Outputs under `ml_model/data/processed/labels_v4/`: `taxonomy_v4.json`
(taxonomy + app display-name mapping), `transactions_labelled_v4.parquet`,
`label_quality_report_v4.json|.md`. Deterministic, streaming, read-only inputs.

In [ ]:
from __future__ import annotations

import json, logging, os, time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

SEED = 42
np.random.seed(SEED)
BATCH_SIZE = 1_048_576
MAX_BATCHES = int(os.environ.get("GALAHAD_V4_MAX_BATCHES", "0")) or None


def find_ml_model_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if cand.name == "ml_model" and (cand / "data" / "processed").is_dir():
            return cand
        if (cand / "ml_model" / "data" / "processed").is_dir():
            return cand / "ml_model"
    raise FileNotFoundError(start)


ML_MODEL_ROOT = find_ml_model_root(Path.cwd().resolve())
SRC = ML_MODEL_ROOT / "data" / "processed" / "labels_v3" / "transactions_labelled_v3.parquet"
assert SRC.is_file(), "run notebook 03 first"
OUT_DIR = ML_MODEL_ROOT / "data" / "processed" / "labels_v4"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PARQUET = OUT_DIR / "transactions_labelled_v4.parquet"

log = logging.getLogger("stage1b_v4")
log.setLevel(logging.INFO)
log.handlers.clear()
fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
for h in (logging.StreamHandler(), logging.FileHandler(OUT_DIR / "labels_v4_run.log", mode="w")):
    h.setFormatter(fmt); log.addHandler(h)
if MAX_BATCHES: log.warning("SMOKE-TEST MODE: %d batches", MAX_BATCHES)

## 1. PFC taxonomy + v3→PFC mapping

`PFC` below is the Plaid Personal Finance Category taxonomy (primary →
detailed). The v3→PFC map is keyed by `(category_v3, subcategory_v3)`, with
`*` as the subcategory wildcard; direction-dependent entries use a
`("@credit"/"@debit")` suffix resolved per row.

In [ ]:
PFC = {
    "INCOME": ["DIVIDENDS", "INTEREST_EARNED", "RETIREMENT_PENSION", "TAX_REFUND",
               "WAGES", "OTHER_INCOME"],
    "TRANSFER_IN": ["CASH_ADVANCES_AND_LOANS", "DEPOSIT",
                    "INVESTMENT_AND_RETIREMENT_FUNDS", "SAVINGS",
                    "ACCOUNT_TRANSFER", "OTHER_TRANSFER_IN"],
    "TRANSFER_OUT": ["INVESTMENT_AND_RETIREMENT_FUNDS", "SAVINGS", "WITHDRAWAL",
                     "ACCOUNT_TRANSFER", "OTHER_TRANSFER_OUT"],
    "LOAN_PAYMENTS": ["CAR_PAYMENT", "CREDIT_CARD_PAYMENT", "MORTGAGE_PAYMENT",
                      "PERSONAL_LOAN_PAYMENT", "STUDENT_LOAN_PAYMENT", "OTHER_PAYMENT"],
    "BANK_FEES": ["ATM_FEES", "FOREIGN_TRANSACTION_FEES", "INSUFFICIENT_FUNDS",
                  "INTEREST_CHARGE", "OVERDRAFT_FEES", "OTHER_BANK_FEES"],
    "ENTERTAINMENT": ["CASINOS_AND_GAMBLING", "MUSIC_AND_AUDIO",
                      "SPORTING_EVENTS_AMUSEMENT_PARKS_AND_MUSEUMS",
                      "TV_AND_MOVIES", "VIDEO_GAMES", "OTHER_ENTERTAINMENT"],
    "FOOD_AND_DRINK": ["BEER_WINE_AND_LIQUOR", "COFFEE", "FAST_FOOD", "GROCERIES",
                       "RESTAURANT", "VENDING_MACHINES", "OTHER_FOOD_AND_DRINK"],
    "GENERAL_MERCHANDISE": ["BOOKSTORES_AND_NEWSSTANDS", "CLOTHING_AND_ACCESSORIES",
                            "CONVENIENCE_STORES", "DEPARTMENT_STORES", "DISCOUNT_STORES",
                            "ELECTRONICS", "GIFTS_AND_NOVELTIES", "OFFICE_SUPPLIES",
                            "ONLINE_MARKETPLACES", "PET_SUPPLIES", "SPORTING_GOODS",
                            "SUPERSTORES", "TOBACCO_AND_VAPE", "OTHER_GENERAL_MERCHANDISE"],
    "HOME_IMPROVEMENT": ["FURNITURE", "HARDWARE", "REPAIR_AND_MAINTENANCE",
                         "SECURITY", "OTHER_HOME_IMPROVEMENT"],
    "MEDICAL": ["DENTAL_CARE", "EYE_CARE", "NURSING_CARE",
                "PHARMACIES_AND_SUPPLEMENTS", "PRIMARY_CARE",
                "VETERINARY_SERVICES", "OTHER_MEDICAL"],
    "PERSONAL_CARE": ["GYMS_AND_FITNESS_CENTERS", "HAIR_AND_BEAUTY",
                      "LAUNDRY_AND_DRY_CLEANING", "OTHER_PERSONAL_CARE"],
    "GENERAL_SERVICES": ["ACCOUNTING_AND_FINANCIAL_PLANNING", "AUTOMOTIVE",
                         "CHILDCARE", "CONSULTING_AND_LEGAL", "EDUCATION",
                         "INSURANCE", "POSTAGE_AND_SHIPPING", "STORAGE",
                         "OTHER_GENERAL_SERVICES"],
    "GOVERNMENT_AND_NON_PROFIT": ["DONATIONS", "GOVERNMENT_DEPARTMENTS_AND_AGENCIES",
                                  "TAX_PAYMENT", "OTHER_GOVERNMENT_AND_NON_PROFIT"],
    "TRANSPORTATION": ["BIKES_AND_SCOOTERS", "GAS", "PARKING", "PUBLIC_TRANSIT",
                       "TAXIS_AND_RIDE_SHARES", "TOLLS", "OTHER_TRANSPORTATION"],
    "TRAVEL": ["FLIGHTS", "LODGING", "RENTAL_CARS", "OTHER_TRAVEL"],
    "RENT_AND_UTILITIES": ["GAS_AND_ELECTRICITY", "INTERNET_AND_CABLE", "RENT",
                           "SEWAGE_AND_WASTE_MANAGEMENT", "TELEPHONE", "WATER",
                           "OTHER_UTILITIES"],
    "UNCATEGORIZED": [],
}

# (category_v3, subcategory_v3 or '*') -> (primary, detailed); '@credit'/'@debit'
# key suffix marks direction-specific rows.
M = {
    ("income", "salary"): ("INCOME", "WAGES"),
    ("income", "business_income"): ("INCOME", "OTHER_INCOME"),
    ("income", "pension_benefits"): ("INCOME", "RETIREMENT_PENSION"),
    ("income", "refunds_reimbursements"): ("INCOME", "OTHER_INCOME"),
    ("income", "interest_income"): ("INCOME", "INTEREST_EARNED"),
    ("income", "*"): ("INCOME", "OTHER_INCOME"),
    ("transfers", "send_money@debit"): ("TRANSFER_OUT", "OTHER_TRANSFER_OUT"),
    ("transfers", "send_money@credit"): ("TRANSFER_IN", "OTHER_TRANSFER_IN"),
    ("transfers", "received_money"): ("TRANSFER_IN", "OTHER_TRANSFER_IN"),
    ("transfers", "wallet_voucher@debit"): ("TRANSFER_OUT", "OTHER_TRANSFER_OUT"),
    ("transfers", "wallet_voucher@credit"): ("TRANSFER_IN", "OTHER_TRANSFER_IN"),
    ("transfers", "*@debit"): ("TRANSFER_OUT", "ACCOUNT_TRANSFER"),
    ("transfers", "*@credit"): ("TRANSFER_IN", "ACCOUNT_TRANSFER"),
    ("cash", "atm_withdrawal"): ("TRANSFER_OUT", "WITHDRAWAL"),
    ("cash", "cash_deposit"): ("TRANSFER_IN", "DEPOSIT"),
    ("cash", "*@debit"): ("TRANSFER_OUT", "WITHDRAWAL"),
    ("cash", "*@credit"): ("TRANSFER_IN", "DEPOSIT"),
    ("bank_fees", "account_maintenance"): ("BANK_FEES", "OTHER_BANK_FEES"),
    ("bank_fees", "penalty_fees"): ("BANK_FEES", "OVERDRAFT_FEES"),
    ("bank_fees", "taxes_duties"): ("BANK_FEES", "OTHER_BANK_FEES"),
    ("bank_fees", "*"): ("BANK_FEES", "OTHER_BANK_FEES"),
    ("loans_credit", "personal_loan@credit"): ("TRANSFER_IN", "CASH_ADVANCES_AND_LOANS"),
    ("loans_credit", "microloan@credit"): ("TRANSFER_IN", "CASH_ADVANCES_AND_LOANS"),
    ("loans_credit", "personal_loan"): ("LOAN_PAYMENTS", "PERSONAL_LOAN_PAYMENT"),
    ("loans_credit", "home_loan"): ("LOAN_PAYMENTS", "MORTGAGE_PAYMENT"),
    ("loans_credit", "vehicle_finance"): ("LOAN_PAYMENTS", "CAR_PAYMENT"),
    ("loans_credit", "credit_card"): ("LOAN_PAYMENTS", "CREDIT_CARD_PAYMENT"),
    ("loans_credit", "store_credit"): ("LOAN_PAYMENTS", "OTHER_PAYMENT"),
    ("loans_credit", "debt_repayment"): ("LOAN_PAYMENTS", "OTHER_PAYMENT"),
    ("loans_credit", "microloan"): ("LOAN_PAYMENTS", "PERSONAL_LOAN_PAYMENT"),
    ("loans_credit", "*@credit"): ("TRANSFER_IN", "CASH_ADVANCES_AND_LOANS"),
    ("loans_credit", "*"): ("LOAN_PAYMENTS", "OTHER_PAYMENT"),
    ("insurance", "*"): ("GENERAL_SERVICES", "INSURANCE"),
    ("groceries", "agricultural_retail"): ("GENERAL_MERCHANDISE", "OTHER_GENERAL_MERCHANDISE"),
    ("groceries", "*"): ("FOOD_AND_DRINK", "GROCERIES"),
    ("food_drink", "fast_food"): ("FOOD_AND_DRINK", "FAST_FOOD"),
    ("food_drink", "restaurant"): ("FOOD_AND_DRINK", "RESTAURANT"),
    ("food_drink", "cafe"): ("FOOD_AND_DRINK", "COFFEE"),
    ("food_drink", "bars_nightlife"): ("FOOD_AND_DRINK", "BEER_WINE_AND_LIQUOR"),
    ("food_drink", "alcohol_liquor"): ("FOOD_AND_DRINK", "BEER_WINE_AND_LIQUOR"),
    ("food_drink", "*"): ("FOOD_AND_DRINK", "OTHER_FOOD_AND_DRINK"),
    ("shopping", "clothing_accessories"): ("GENERAL_MERCHANDISE", "CLOTHING_AND_ACCESSORIES"),
    ("shopping", "electronics"): ("GENERAL_MERCHANDISE", "ELECTRONICS"),
    ("shopping", "furniture_appliances"): ("HOME_IMPROVEMENT", "FURNITURE"),
    ("shopping", "building_hardware"): ("HOME_IMPROVEMENT", "HARDWARE"),
    ("shopping", "online_shopping"): ("GENERAL_MERCHANDISE", "ONLINE_MARKETPLACES"),
    ("shopping", "*"): ("GENERAL_MERCHANDISE", "OTHER_GENERAL_MERCHANDISE"),
    ("transport_fuel", "fuel"): ("TRANSPORTATION", "GAS"),
    ("transport_fuel", "vehicle_maintenance"): ("GENERAL_SERVICES", "AUTOMOTIVE"),
    ("transport_fuel", "public_transport"): ("TRANSPORTATION", "PUBLIC_TRANSIT"),
    ("transport_fuel", "car_rental"): ("TRAVEL", "RENTAL_CARS"),
    ("transport_fuel", "parking_tolls"): ("TRANSPORTATION", "PARKING"),
    ("transport_fuel", "*"): ("TRANSPORTATION", "OTHER_TRANSPORTATION"),
    ("housing_utilities", "rent"): ("RENT_AND_UTILITIES", "RENT"),
    ("housing_utilities", "electricity_water"): ("RENT_AND_UTILITIES", "GAS_AND_ELECTRICITY"),
    ("housing_utilities", "internet_tv"): ("RENT_AND_UTILITIES", "INTERNET_AND_CABLE"),
    ("housing_utilities", "*"): ("RENT_AND_UTILITIES", "OTHER_UTILITIES"),
    ("communication", "*"): ("RENT_AND_UTILITIES", "TELEPHONE"),
    ("health_beauty", "pharmacy"): ("MEDICAL", "PHARMACIES_AND_SUPPLEMENTS"),
    ("health_beauty", "medical"): ("MEDICAL", "PRIMARY_CARE"),
    ("health_beauty", "fitness"): ("PERSONAL_CARE", "GYMS_AND_FITNESS_CENTERS"),
    ("health_beauty", "*"): ("PERSONAL_CARE", "OTHER_PERSONAL_CARE"),
    ("education", "*"): ("GENERAL_SERVICES", "EDUCATION"),
    ("entertainment_leisure", "subscriptions_streaming"): ("ENTERTAINMENT", "TV_AND_MOVIES"),
    ("entertainment_leisure", "betting_gaming"): ("ENTERTAINMENT", "CASINOS_AND_GAMBLING"),
    ("entertainment_leisure", "events_travel"): ("TRAVEL", "OTHER_TRAVEL"),
    ("entertainment_leisure", "hobbies_outdoors"): ("GENERAL_MERCHANDISE", "SPORTING_GOODS"),
    ("entertainment_leisure", "*"): ("ENTERTAINMENT", "OTHER_ENTERTAINMENT"),
    ("savings_investments", "savings@debit"): ("TRANSFER_OUT", "SAVINGS"),
    ("savings_investments", "savings@credit"): ("TRANSFER_IN", "SAVINGS"),
    ("savings_investments", "*@debit"): ("TRANSFER_OUT", "INVESTMENT_AND_RETIREMENT_FUNDS"),
    ("savings_investments", "*@credit"): ("TRANSFER_IN", "INVESTMENT_AND_RETIREMENT_FUNDS"),
    ("services_business", "security_services"): ("HOME_IMPROVEMENT", "SECURITY"),
    ("services_business", "*"): ("GENERAL_SERVICES", "OTHER_GENERAL_SERVICES"),
    ("uncategorised", "*"): ("UNCATEGORIZED", None),
}

# app-facing display names (detailed first, then primary fallback)
APP_DISPLAY = {
    "primary": {
        "INCOME": "Income", "TRANSFER_IN": "Transfers", "TRANSFER_OUT": "Transfers",
        "LOAN_PAYMENTS": "Loan", "BANK_FEES": "Bank Fees",
        "ENTERTAINMENT": "Entertainment", "FOOD_AND_DRINK": "Eating Out",
        "GENERAL_MERCHANDISE": "Shopping", "HOME_IMPROVEMENT": "Shopping",
        "MEDICAL": "Healthcare", "PERSONAL_CARE": "Health & Beauty",
        "GENERAL_SERVICES": "Services", "GOVERNMENT_AND_NON_PROFIT": "Services",
        "TRANSPORTATION": "Fuel/Retail", "TRAVEL": "Travel",
        "RENT_AND_UTILITIES": "Utilities", "UNCATEGORIZED": "Uncategorized",
    },
    "detailed": {
        "WAGES": "Salary", "RETIREMENT_PENSION": "Pension Benefit",
        "WITHDRAWAL": "Cash", "DEPOSIT": "Deposit",
        "SAVINGS": "Savings", "INVESTMENT_AND_RETIREMENT_FUNDS": "Investments",
        "CASH_ADVANCES_AND_LOANS": "Loan",
        "MORTGAGE_PAYMENT": "Home Loan", "CAR_PAYMENT": "Vehicle Finance",
        "CREDIT_CARD_PAYMENT": "Credit Card Payment",
        "PERSONAL_LOAN_PAYMENT": "Personal / Business Loan",
        "INSURANCE": "Insurance", "GROCERIES": "Groceries",
        "FAST_FOOD": "Fast Food", "RESTAURANT": "Eating Out",
        "COFFEE": "Coffee Shops/Cafes", "BEER_WINE_AND_LIQUOR": "Liquor Stores/Alcohol",
        "CLOTHING_AND_ACCESSORIES": "Shopping", "ELECTRONICS": "Shopping",
        "HARDWARE": "Building Supplies", "GAS": "Fuel/Retail",
        "TELEPHONE": "Airtime", "PHARMACIES_AND_SUPPLEMENTS": "Health & Beauty",
        "PRIMARY_CARE": "Healthcare", "EDUCATION": "Education",
        "TV_AND_MOVIES": "Subscriptions", "CASINOS_AND_GAMBLING": "Entertainment",
        "INSUFFICIENT_FUNDS": "Insufficient Funds",
    },
}

for (c, s), (p, d) in M.items():
    assert p in PFC, (c, s, p)
    assert d is None or d in PFC[p], (c, s, d)
SPEND_PRIMARIES = {"FOOD_AND_DRINK", "GENERAL_MERCHANDISE", "HOME_IMPROVEMENT",
                   "MEDICAL", "PERSONAL_CARE", "GENERAL_SERVICES", "ENTERTAINMENT",
                   "TRANSPORTATION", "TRAVEL", "RENT_AND_UTILITIES"}
print(f"mapping entries: {len(M)}; PFC: {len(PFC) - 1} primaries, "
      f"{sum(len(v) for v in PFC.values())} detailed")

## 2. Streaming pass — write labels v4

In [ ]:
def resolve(cat: str, sub: str, direction: str):
    sub = sub if isinstance(sub, str) and sub else "*"
    dsuf = "@credit" if direction == "c" else "@debit"
    for key in ((cat, sub + dsuf), (cat, sub), (cat, "*" + dsuf), (cat, "*")):
        if key in M:
            return M[key]
    return ("UNCATEGORIZED", None)


# build a lookup for every (cat, sub, dir) combo present — resolved once, then
# applied vectorised per batch
lookup_cache: dict = {}

pf = pq.ParquetFile(SRC)
writer = None
prim_counter, det_counter, flow_counter = Counter(), Counter(), Counter()
refund_count = rows = 0
t0 = time.time()

for bi, batch in enumerate(pf.iter_batches(batch_size=BATCH_SIZE)):
    if MAX_BATCHES is not None and bi >= MAX_BATCHES:
        break
    df = batch.to_pandas()
    cat = df["category_v3"].fillna("uncategorised")
    sub = df["subcategory_v3"].fillna("")
    dc = df["TransactionTypeDebitCredit"].fillna("d")

    keys = pd.Series(list(zip(cat, sub, dc)), index=df.index)
    uniq = keys.unique()
    for k in uniq:
        if k not in lookup_cache:
            lookup_cache[k] = resolve(*k)
    mapped = keys.map(lookup_cache)
    df["pfc_primary"] = mapped.str[0]
    df["pfc_detailed"] = mapped.str[1]
    df["flow"] = np.where(dc == "c", "inflow", "outflow")
    df["possible_refund"] = df["pfc_primary"].isin(SPEND_PRIMARIES) & (dc == "c")

    table = pa.Table.from_pandas(df, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUT_PARQUET, table.schema, compression="snappy")
    else:
        table = table.cast(writer.schema)
    writer.write_table(table)

    rows += len(df)
    prim_counter.update(df["pfc_primary"].value_counts().to_dict())
    det_counter.update(df["pfc_detailed"].dropna().value_counts().to_dict())
    flow_counter.update(df["flow"].value_counts().to_dict())
    refund_count += int(df["possible_refund"].sum())
    if (bi + 1) % 6 == 0:
        log.info("batch %d — %s rows (%.0fs)", bi + 1, f"{rows:,}", time.time() - t0)

if writer is not None:
    writer.close()
assert pq.ParquetFile(OUT_PARQUET).metadata.num_rows == rows
log.info("wrote %s rows -> %s (%.0f MB, %.0fs)", f"{rows:,}", OUT_PARQUET.name,
         OUT_PARQUET.stat().st_size / 1e6, time.time() - t0)

## 3. Quality report & taxonomy artifact

In [ ]:
labelled = rows - prim_counter.get("UNCATEGORIZED", 0)
taxonomy_v4 = {
    "version": "4.0",
    "taxonomy_family": "Plaid Personal Finance Categories aligned",
    "plaid_pfc_version": "v1",
    "internal_taxonomy_version": "galahad_v4_2026_07",
    "created": datetime.now(timezone.utc).isoformat(),
    "flow_values": ["inflow", "outflow"],
    "status_values": ["normal", "reversal", "unpaid", "insufficient_funds"],
    "categories": PFC,
    "app_display": APP_DISPLAY,
}
(OUT_DIR / "taxonomy_v4.json").write_text(json.dumps(taxonomy_v4, indent=2))

report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "rows": rows,
    "smoke_test_mode": MAX_BATCHES is not None,
    "labelled_rate": round(labelled / rows, 4),
    "flow_distribution": dict(flow_counter),
    "possible_refund_rows": refund_count,
    "primary_distribution": {k: int(v) for k, v in
                             sorted(prim_counter.items(), key=lambda x: -x[1])},
    "detailed_distribution": {k: int(v) for k, v in
                              sorted(det_counter.items(), key=lambda x: -x[1])},
}
(OUT_DIR / "label_quality_report_v4.json").write_text(json.dumps(report, indent=2))
md_lines = [
    "# Labels v4 (PFC) report",
    f"- rows: {rows:,}; labelled: {labelled / rows:.1%}",
    f"- flow: {dict(flow_counter)}",
    f"- possible refunds (spend + credit): {refund_count:,}",
    "", "## Primary distribution",
] + [f"- {k}: {v:,}" for k, v in sorted(prim_counter.items(), key=lambda x: -x[1])]
(OUT_DIR / "label_quality_report_v4.md").write_text("\n".join(md_lines))
log.info("labelled rate %.1f%%", 100 * labelled / rows)
print(json.dumps({k: report[k] for k in
                  ["rows", "labelled_rate", "flow_distribution", "possible_refund_rows"]},
                 indent=2))
pd.Series(prim_counter).sort_values(ascending=False)

## 4. Result

`labels_v4/transactions_labelled_v4.parquet` carries, per row: `pfc_primary`
(Plaid PFCv1-aligned and direction-aware), `pfc_detailed`, `flow`
(inflow/outflow), `possible_refund`, plus everything from v3 (merchant_v3,
status, template, provenance). `taxonomy_v4.json` includes the app display-name
mapping so the frontend can adopt PFC gradually. Next:
`06_train_pfc_model.ipynb` retrains the hybrid model on `pfc_primary`.